# matching-pmh: Hospital A → Hospital B (5 minutes)

**Story:** You trained a classifier on **Hospital A** scans. You deploy at **Hospital B** (different camera/lighting). Labels still mean the same disease.

This notebook uses **synthetic** data so nothing is downloaded. You will see **target accuracy** with and without PMH.

Docs: [What is PMH?](https://github.com/vishalstark512/matching-pmh/blob/main/docs/WHAT_IS_PMH.md) · [First hour](https://github.com/vishalstark512/matching-pmh/blob/main/docs/FIRST_HOUR.md)

In [ ]:
!pip install -q matching-pmh torch

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from pmh import PMHConfig, PMHTrainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)


class Backbone(nn.Module):
    def __init__(self, d_in=32, d=16):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.ReLU(), nn.Linear(d, d))

    def forward(self, x):
        return self.net(x)


def make_loader(n, batch, shift, seed):
    g = torch.Generator().manual_seed(seed)
    x = torch.randn(n, 32, generator=g) + shift
    y = torch.randint(0, 2, (n,), generator=g)
    return DataLoader(TensorDataset(x, y), batch_size=batch, shuffle=True)


n, batch, epochs = 400, 32, 8
hospital_a = make_loader(n, batch, shift=0.0, seed=1)   # source
hospital_b = make_loader(n, batch, shift=0.8, seed=2)   # target (deploy)
train_mix = make_loader(n, batch, shift=0.2, seed=3)
hospital_b_test = make_loader(n // 2, batch, shift=0.8, seed=4)

print("Synthetic loaders ready (Hospital A shift=0, B shift=0.8)")

In [ ]:
@torch.no_grad()
def target_accuracy(model, encoder, head, loader):
    model.eval()
    ok = tot = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = head(encoder(xb)).argmax(1)
        ok += (pred == yb).sum().item()
        tot += yb.numel()
    return ok / max(tot, 1)


def train_erm(model, loader, epochs, lr=1e-2):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()


backbone = Backbone().to(device)
head = nn.Linear(16, 2).to(device)
baseline = nn.Sequential(backbone, head).to(device)

train_erm(baseline, train_mix, epochs=epochs)
acc_baseline = target_accuracy(baseline, backbone, head, hospital_b_test)
print(f"Target accuracy (baseline ERM): {acc_baseline:.3f}")

In [ ]:
backbone_pmh = Backbone().to(device)
head_pmh = nn.Linear(16, 2).to(device)
model_pmh = nn.Sequential(backbone_pmh, head_pmh).to(device)

trainer = PMHTrainer(
    model_pmh,
    hook=backbone_pmh,
    head=head_pmh,
    nuisance="domain_shift",
    rank=6,
    pmh_config=PMHConfig.balanced(),
    device=device,
)
trainer.fit(
    train_mix,
    source_batches=hospital_a,
    target_batches=hospital_b,
    epochs=epochs,
)

acc_pmh = target_accuracy(model_pmh, backbone_pmh, head_pmh, hospital_b_test)
print(f"Target accuracy (with PMH):      {acc_pmh:.3f}")
print(f"Preflight check: {trainer.artifact_.preflight}")

## What next?

1. Replace synthetic loaders with **your** Hospital A / B `DataLoader`s (same label semantics).
2. Point `hook=` at your CNN/ViT backbone (see [hook cookbook](https://github.com/vishalstark512/matching-pmh/blob/main/docs/hooks.md)).
3. Before production claims, run [falsification controls](https://github.com/vishalstark512/matching-pmh/blob/main/docs/walkthroughs/08-falsification-controls.md).

```python
from pmh.onboarding import print_setup_guide
print_setup_guide(stack="pytorch", has_target_domain=True, has_target_labels=False)
```